# F7 homework: an end-to-end tabular classification project

A phone company wants to know which customers are likely to **cancel their contract (churn)** in the
next three months, so its retention team can contact them first. You will build that model from start
to finish, following the workflow from the last lesson of F7:

1. **Explore** the data and spot a leaky column.
2. **Split** off a test set and leave it alone.
3. **Preprocess** mixed numeric and categorical columns with missing values, in a `ColumnTransformer`.
4. **Compare** a baseline, an unpenalised and a regularised logistic regression, and k-NN, with
   cross-validation and `GridSearchCV`.
5. **Choose a threshold** that meets the business requirement, on cross-validated predictions.
6. **Evaluate once** on the test set.
7. **Write up** what you found.

The dataset is synthetic, generated by a cell below with a fixed seed, so the notebook runs offline
and everyone gets the same numbers. It is modelled on real churn data: some columns matter, one is
noise, some values are missing, and one column leaks the answer.

**Setup.** Any laptop or free Colab (CPU), with scikit-learn 1.2 or newer, pandas and matplotlib. Every
cell runs in seconds. Each task has `TODO` cells and a check cell that prints `OK` when your answer is
right. Run the cells in order; if you change something early on, use *Restart and run all*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix
from sklearn.base import clone
import sklearn
print("scikit-learn", sklearn.__version__)

## The data (given)

One row per customer, 3,000 customers:

| Column | Meaning |
|---|---|
| `tenure_months` | months since the customer joined |
| `monthly_charge` | monthly bill in £ (some missing) |
| `support_calls` | calls to customer support in the last 3 months |
| `contract` | `month-to-month`, `one-year` or `two-year` |
| `payment_method` | `card`, `bank transfer`, `electronic cheque` or `post` (some missing) |
| `region` | one of five sales regions |
| `retention_call` | 1 if the retention team phoned the customer. The team phones customers **after they ask to cancel**. |
| `churned` | **the label**: 1 if the customer cancelled within three months |

Run the cell; do not change it.

In [ ]:
rng = np.random.default_rng(7)
n = 3000
tenure = rng.integers(1, 73, n)
monthly = np.round(np.clip(rng.normal(65, 25, n), 15, 150), 2)
calls = rng.poisson(1.5, n)
contract = rng.choice(["month-to-month", "one-year", "two-year"], n, p=[0.55, 0.25, 0.20])
payment = rng.choice(["card", "bank transfer", "electronic cheque", "post"], n, p=[0.35, 0.25, 0.3, 0.1])
region = rng.choice(["north", "south", "east", "west", "central"], n)
logit = (-1.0 - 0.045 * tenure + 0.02 * (monthly - 65) + 0.5 * calls
         + np.select([contract == "month-to-month", contract == "two-year"], [1.2, -1.3], 0.0)
         + 0.6 * (payment == "electronic cheque"))
churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
retention_call = np.where(churn == 1, rng.random(n) < 0.8, rng.random(n) < 0.05).astype(int)
df = pd.DataFrame({"tenure_months": tenure, "monthly_charge": monthly, "support_calls": calls,
                   "contract": contract, "payment_method": payment, "region": region,
                   "retention_call": retention_call, "churned": churn})
df.loc[rng.random(n) < 0.04, "monthly_charge"] = np.nan
df.loc[rng.random(n) < 0.03, "payment_method"] = np.nan
df.head()

## Task 1: explore the data

**1a.** Compute:

- `churn_rate`: the share of customers who churned (a number between 0 and 1);
- `missing`: a pandas Series with the number of missing values in each column (`df.isna().sum()`);
- `by_contract`: a Series with the churn rate for each contract type (use `groupby`, F2).

In [ ]:
churn_rate = None   # TODO
missing = None      # TODO
by_contract = None  # TODO

print(f"churn rate {churn_rate}")
print(missing)
print(by_contract)

In [ ]:
assert churn_rate is not None and abs(churn_rate - df["churned"].mean()) < 1e-9, "churn_rate should be the mean of the churned column"
assert missing is not None and int(missing["monthly_charge"]) == int(df["monthly_charge"].isna().sum()), "missing should count NaNs per column: df.isna().sum()"
assert by_contract is not None and set(by_contract.index) == {"month-to-month", "one-year", "two-year"}, "by_contract should have one entry per contract type"
assert abs(by_contract["two-year"] - df.loc[df["contract"] == "two-year", "churned"].mean()) < 1e-9, "by_contract should hold the churn rate (mean of churned) per contract"
print("OK")

**1b.** Make a figure with two panels: a bar chart of `by_contract`, and histograms of `tenure_months`
for churned and non-churned customers on the same axes (use `alpha=0.5` and `density=True` so the two
groups are comparable). Label the axes and add a legend. What do the two panels suggest?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
# TODO: left panel, churn rate by contract; right panel, tenure histograms by churn
plt.tight_layout()
plt.show()

**1c. Leakage.** Read the column table again. One column would **not be known at the moment the model
must make its prediction**. Measure how well that column predicts churn on its own: the mean 5-fold
cross-validated ROC AUC of a `LogisticRegression` fitted on just that column (`df[[column]]`). Store it
in `leak_auc` and the column name in `leaky_column`.

Compare `leak_auc` with the scores you get in Task 4. In a sentence in the write-up: why must this
column be dropped, even though it is the best predictor?

In [ ]:
leaky_column = None  # TODO: the column name, as a string
leak_auc = None      # TODO
print(leaky_column, leak_auc)

In [ ]:
assert leaky_column == "retention_call", "Which column is only filled in after the customer has asked to cancel?"
assert leak_auc is not None and abs(leak_auc - cross_val_score(LogisticRegression(), df[["retention_call"]], df["churned"], cv=5, scoring="roc_auc").mean()) < 1e-6, "leak_auc: cross_val_score(LogisticRegression(), df[[leaky_column]], df['churned'], cv=5, scoring='roc_auc').mean()"
print("OK")

## Task 2: split off the test set

Build `X` (every column except `churned` and the leaky column) and `y` (the `churned` column as a NumPy
array). Then split 80/20 with `train_test_split(..., test_size=0.2, stratify=y, random_state=0)` into
`X_train, X_test, y_train, y_test`. From now on, **do not look at `X_test` or `y_test` until Task 6.**

In [ ]:
X = None  # TODO
y = None  # TODO
X_train, X_test, y_train, y_test = None, None, None, None  # TODO

In [ ]:
assert X is not None and "churned" not in X.columns and "retention_call" not in X.columns, "X must not contain the label or the leaky column"
assert X.shape == (3000, 6), f"X should have 6 feature columns, got shape {X.shape}"
assert X_train is not None and len(X_train) == 2400 and len(X_test) == 600, "use test_size=0.2"
assert abs(np.mean(y_train) - np.mean(y_test)) < 0.005, "use stratify=y so both sets have the same churn rate"
assert list(X_train.index[:3]) == list(train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)[0].index[:3]), "use random_state=0"
print("OK")

## Task 3: preprocessing

Create `numeric` and `categorical`, the lists of numeric and categorical column names in `X`. Then
build `preprocess`, a `ColumnTransformer` that sends

- the numeric columns through `SimpleImputer(strategy="median")` then `StandardScaler()`;
- the categorical columns through `SimpleImputer(strategy="most_frequent")` then
  `OneHotEncoder(handle_unknown="ignore")`.

List the numeric transformer first (the check expects the 3 numeric columns first). Do **not** fit it here: it goes inside every model pipeline, so it is refitted on each training fold.

In [ ]:
numeric = None     # TODO
categorical = None # TODO
preprocess = None  # TODO

In [ ]:
assert numeric is not None and sorted(numeric) == ["monthly_charge", "support_calls", "tenure_months"], "numeric should list the three numeric columns"
assert categorical is not None and sorted(categorical) == ["contract", "payment_method", "region"], "categorical should list the three text columns"
_Z = clone(preprocess).fit_transform(X_train)
_Z = _Z.toarray() if hasattr(_Z, "toarray") else np.asarray(_Z)
assert _Z.shape == (2400, 15), f"expected 3 numeric + 3 + 4 + 5 one-hot = 15 columns, got {_Z.shape}"
assert not np.isnan(_Z).any(), "there are still missing values: check the imputers"
assert np.allclose(_Z[:, :3].mean(axis=0), 0, atol=1e-8), "numeric columns should come first and be standardised (mean 0) after imputation"
print("OK")

## Task 4: compare models with cross-validation

Use the same folds for every model: `cv` is given below. Score everything with ROC AUC. Fill the
dictionary `results` with `name: (mean AUC, std AUC)` for four models, each a `Pipeline` of
`("prep", preprocess)` and `("clf", ...)` (except the dummy, which needs no preprocessing):

1. `"dummy"`: `DummyClassifier()`, with `cross_val_score`;
2. `"logistic"`: `LogisticRegression(C=1e4, max_iter=1000)`, almost unpenalised, with `cross_val_score`;
3. `"ridge_logistic"`: `LogisticRegression(max_iter=1000)` with `GridSearchCV` over
   `{"clf__C": [0.001, 0.01, 0.1, 1, 10]}`; store the fitted search in `search_lr`;
4. `"knn"`: `KNeighborsClassifier()` with `GridSearchCV` over
   `{"clf__n_neighbors": [5, 25, 51, 101, 201]}`; store it in `search_knn`.

For a grid search, the mean is `search.best_score_` and the std is
`search.cv_results_["std_test_score"][search.best_index_]`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
results = {}
search_lr = None   # TODO
search_knn = None  # TODO
# TODO: fill results

In [ ]:
if results:
    table = pd.DataFrame(results, index=["mean AUC", "std"]).T.sort_values("mean AUC", ascending=False)
    print(table.round(4))
if search_lr is not None and search_knn is not None:
    print("best C:", search_lr.best_params_, "  best k:", search_knn.best_params_)

In [ ]:
assert set(results) == {"dummy", "logistic", "ridge_logistic", "knn"}, f"results needs the four model names, has {sorted(results)}"
assert abs(results["dummy"][0] - 0.5) < 1e-9, "a dummy classifier ranks at random: AUC 0.5"
assert results["logistic"][0] > 0.8, "the logistic pipeline should reach a CV AUC above 0.8: is preprocess inside the pipeline?"
assert search_lr is not None and search_lr.best_params_["clf__C"] in [0.001, 0.01, 0.1, 1, 10], "search_lr should be a fitted GridSearchCV over clf__C"
assert search_knn is not None and "clf__n_neighbors" in search_knn.best_params_, "search_knn should be a fitted GridSearchCV over clf__n_neighbors"
assert abs(results["knn"][0] - search_knn.best_score_) < 1e-12, "for knn, use search_knn.best_score_ as the mean"
print("OK")

**Questions for the write-up.** Which model wins? Is the gap bigger than the fold-to-fold standard
deviation? Why does regularisation barely change the logistic regression here? (Think about how many
rows there are per feature, and lesson 3's discussion of variance.) Is the best `k` for k-NN at the edge
of its grid, and what would you do if it were?

## Task 5: choose the decision threshold

The retention team can phone a limited number of customers, but the business requires that the
model flags **at least 70% of the customers who will churn** (recall of at least 0.7). Among thresholds
that meet this, the highest one flags the fewest customers, so it is the cheapest.

Use the best model from Task 4, `best_model = search_lr.best_estimator_` (or `search_knn`'s if k-NN won).
Never choose the threshold on the test set. Instead:

1. Compute `oof`, the **out-of-fold** probabilities on the training set:
   `cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]`. Each row's
   probability comes from a model that did not see that row.
2. Write `threshold_for_recall(y_true, probs, target)` (F7 lesson 6, exercise 3) and set
   `threshold = threshold_for_recall(y_train, oof, 0.7)`.
3. Plot precision and recall on `oof` against the threshold (for thresholds from 0.05 to 0.95), with a
   vertical line at your threshold.

In [ ]:
best_model = search_lr.best_estimator_  # change if k-NN won
oof = None  # TODO

def threshold_for_recall(y_true, probs, target):
    # TODO: the largest threshold t (one of the probs) with recall(probs >= t) >= target
    pass

threshold = None  # TODO
print("threshold:", threshold)

In [ ]:
ts = np.linspace(0.05, 0.95, 91)
# TODO: compute precision and recall of (oof >= t) for each t in ts, plot both, mark the threshold

In [ ]:
assert oof is not None and np.shape(oof) == (2400,), "oof should hold one out-of-fold probability per training row"
assert np.allclose(oof, cross_val_predict(best_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]), "compute oof with cross_val_predict(best_model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]"
_rec = lambda t: np.sum((oof >= t) & (y_train == 1)) / np.sum(y_train == 1)
assert threshold is not None and _rec(threshold) >= 0.7, "at your threshold the out-of-fold recall is below 0.7"
_higher = np.unique(oof)[np.unique(oof) > threshold]
assert len(_higher) == 0 or _rec(_higher.min()) < 0.7, "a higher threshold also reaches recall 0.7: return the largest one"
print("OK, out-of-fold precision at this threshold:", round(precision_score(y_train, oof >= threshold), 3))

## Task 6: evaluate once on the test set

Fit `best_model` on the whole training set. Then, on the test set, compute `test_auc` (ROC AUC of the
probabilities), and `test_precision`, `test_recall` and `test_cm` (the confusion matrix) for
predictions at your `threshold`. This is the only time the test set is used.

In [ ]:
test_auc = None        # TODO
test_precision = None  # TODO
test_recall = None     # TODO
test_cm = None         # TODO
print(f"test AUC {test_auc}  precision {test_precision}  recall {test_recall}")
print(test_cm)

In [ ]:
_p = clone(best_model).fit(X_train, y_train).predict_proba(X_test)[:, 1]
assert test_auc is not None and abs(test_auc - roc_auc_score(y_test, _p)) < 1e-9, "test_auc: fit best_model on X_train, then roc_auc_score(y_test, probabilities on X_test)"
assert abs(test_recall - recall_score(y_test, _p >= threshold)) < 1e-9, "test_recall should use your threshold on the test probabilities"
assert abs(test_precision - precision_score(y_test, _p >= threshold)) < 1e-9, "test_precision should use your threshold on the test probabilities"
assert np.array_equal(np.asarray(test_cm), confusion_matrix(y_test, (_p >= threshold).astype(int))), "test_cm should be confusion_matrix(y_test, predictions at your threshold)"
print("OK")

Expect test recall a little below 0.7 (about 0.63 here). That is not a bug: with about 170 test churners, recall has a standard error of about 0.035, and test AUC (0.80) is also a bit below CV (0.82). Discuss it in Task 7.

## Task 7: write-up

Replace this cell with a short report (8 to 12 sentences) for the retention team's manager:

- the final model and its test ROC AUC, next to the baseline's;
- at your threshold: what share of churners it catches (recall), and what share of flagged customers
  actually churn (precision), in plain words;
- whether the test recall met the 70% target, and why it can differ from the out-of-fold value
  (how many churners are in the test set?);
- the leaky column, and what would have happened if you had kept it;
- one limitation and one next step.

## Stretch tasks

- **Interactions:** add `PolynomialFeatures(degree=2)` between `prep` and `clf` in the regularised
  logistic regression and search `C` again. Does the extra flexibility help? Does the best `C` change?
- **Feature selection with L1:** fit an L1-penalised logistic regression (in scikit-learn 1.8,
  `LogisticRegression(l1_ratio=1, solver="saga")`; in older versions, `penalty="l1", solver="saga"`).
  Which one-hot columns get zero weights? Is `region` among them?
- **Costs:** suppose a missed churner costs £300 and a phone call £10. Find the threshold that minimises
  the total cost on the out-of-fold predictions, and compare it with your recall-based threshold.
- **Class weights:** refit with `class_weight="balanced"`. How do the AUC, the probabilities and the
  threshold change?